# Create A DuckDB Database and Load Data

In [ ]:
import duckdb
#create duckdb database connection
con = duckdb.connect("../natural_gas.duckdb")


In [ ]:
# Load wide format data into DuckDB
con.execute("""
    CREATE OR REPLACE TABLE gas_wide AS
    SELECT *
    FROM read_csv_auto('../data/processed/consumer_gas_monthly_cleaned.csv')
""")


In [ ]:
# Load schema to get column names
cols = con.execute("PRAGMA table_info('gas_wide')").df()["name"].tolist()
# Prepare list of states for UNPIVOT clause
states = [f'"{c}"' for c in cols if c != "Date"]

# Construct the UNPIVOT clause
unpivot_clause = "UNPIVOT(value FOR state IN (" + ", ".join(states) + "))"

# Create long format table using UNPIVOT
con.execute(f"""
    CREATE OR REPLACE TABLE gas_long AS
    SELECT Date, state, value
    FROM gas_wide
    {unpivot_clause}
""")


In [8]:
con.execute("SELECT * FROM gas_wide LIMIT 5").df()



,Date,US_Total,Alabama,Alaska,Arizona,Arkansas,California,Colorado,Connecticut,Delaware,...,South Dakota,Tennessee,Texas,Utah,Vermont,Virginia,Washington,West Virginia,Wiscons,Wyoming
0,2001-01-01,2505011.0,36984.0,12927.0,19804.0,26139.0,256236.0,57089.0,18442.0,5014.0,...,4302.0,43045.0,362844.0,20043.0,1164.0,34325.0,31231.0,14634.0,52126.0,7475.0
1,2001-02-01,2156873.0,28384.0,11677.0,23088.0,20654.0,225525.0,50447.0,15861.0,4742.0,...,4607.0,30197.0,318006.0,17426.0,1003.0,27001.0,31904.0,12224.0,51020.0,6484.0
2,2001-03-01,2086568.0,27217.0,12492.0,21742.0,21940.0,210711.0,49042.0,16485.0,5389.0,...,4228.0,26202.0,336664.0,13012.0,1084.0,23081.0,29422.0,11221.0,52466.0,5643.0
3,2001-04-01,1663832.0,23714.0,10557.0,19153.0,16528.0,198804.0,41157.0,10646.0,3439.0,...,2845.0,21053.0,325130.0,11173.0,834.0,15728.0,27137.0,9393.0,24969.0,5505.0
4,2001-05-01,1385163.0,21027.0,9618.0,21113.0,13819.0,182600.0,30506.0,7197.0,2924.0,...,1940.0,13399.0,322975.0,7791.0,544.0,11714.0,23855.0,5380.0,17238.0,4182.0


In [9]:
con.execute("SELECT * FROM gas_long LIMIT 5").df()

,Date,state,value
0,2001-01-01,US_Total,2505011.0
1,2001-01-01,Alabama,36984.0
2,2001-01-01,Alaska,12927.0
3,2001-01-01,Arizona,19804.0
4,2001-01-01,Arkansas,26139.0


In [12]:
con.close()